# Experiment — is the Kneedle elbow stable under window length (max BN)?

Same averaged curve (default **P% = 0, BS = 1024**), truncated at different
`max BN` values (default **150, 250, and the full pre-step window**), Kneedle
run on each truncation (`curve='convex'`, `direction='decreasing'`, `S=1.0`).

Why the elbow can move: Kneedle min–max normalizes **within the window** —
x by the window end (max BN), y by the window minimum — so the diagonal and
the difference curve $D=\hat{y}_t-\hat{x}$ are window-dependent. A shorter
window compresses x and raises the y-floor, shifting where $D$ peaks.

Cells: (1) run experiment + result table, (2) knees overlaid on the raw curve,
(3) per-window normalized Kneedle geometry, (4) **min-distance-from-origin
method** (elbow = point closest to the normalized origin; no diagonal, no far
endpoint in the selection rule), (5) window-stability comparison of Kneedle vs
the origin method. Edit `P_LEVEL`, `BS`, `MAX_BN_LIST` in Cell 1.

In [1]:
# ============================================================================
# Cell 1 — EXPERIMENT: Kneedle elbow vs window length (max BN)
#
# Question: for the SAME averaged CE curve, does Kneedle report the same
# elbow when the data is truncated at different max BN?
#
# Kneedle normalizes x and y by the window min/max, so changing max BN
# rescales both axes — the knee is NOT guaranteed to be window-invariant.
# ============================================================================
import os
import numpy as np
import pandas as pd
from kneed import KneeLocator

# ── Experiment config ─────────────────────────────────────────────────────────
P_LEVEL     = 0.0            # pruning level
BS          = 1024           # batch size
MAX_BN_LIST = [150, 250, None]   # window truncations; None = full pre-step window
KNEEDLE_S   = 1.0
KNEEDLE_TAIL_N = None        # None = standard min(y) anchor; int = mean of last N
BN_STEP_MIN = 100
STEP_THRESH = 0.01
CE_o = np.log(10)

BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
# ──────────────────────────────────────────────────────────────────────────────

# Load the averaged curve, truncate at the step artifact (shared convention)
f = os.path.join(BASE_DIR, f"p-percentage_{P_LEVEL}", f"batch_size_{BS}",
                 f"averaged_runs_p_{P_LEVEL}_bs_{BS}.csv")
df = pd.read_csv(f)
df.columns = df.columns.str.strip()
ce_col = next(c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test"))
bn_col = next(c for c in df.columns if "Batch" in c)
df = df.dropna(subset=[ce_col, bn_col])
BN_full = df[bn_col].values.astype(float)
CE_full = df[ce_col].values.astype(float)

cutoff_BN = float(BN_full[-1])
for i in range(1, len(BN_full)):
    if BN_full[i] >= BN_STEP_MIN and abs(CE_full[i] - CE_full[i-1]) > STEP_THRESH:
        cutoff_BN = float(BN_full[i])
        break
m = BN_full < cutoff_BN
BN_base, CE_base = BN_full[m], CE_full[m]
print(f"P%={P_LEVEL*100:.0f}  BS={BS}  step cutoff at BN={cutoff_BN:.0f}  "
      f"({len(BN_base)} points in the pre-step window)")


def kneedle_elbow(BN, CE, S=KNEEDLE_S, tail_n=KNEEDLE_TAIL_N):
    """Kneedle knee on (BN, CE). Returns (knee_BN, knee_CE, IPA) or NaNs."""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < 3 or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    if tail_n is not None:
        k = int(min(tail_n, len(CE)))
        floor = float(np.mean(CE[-k:]))
        if CE[0] - floor <= 1e-10:
            return np.nan, np.nan, np.nan
        y = np.clip(CE, floor, None)
    else:
        y = CE
    kl = KneeLocator(BN, y, curve="convex", direction="decreasing", S=S)
    if kl.knee is None:
        return np.nan, np.nan, np.nan
    i = int(np.argmin(np.abs(BN - kl.knee)))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


# Run the experiment: same data, different max BN
results = []
windows = {}   # max_bn label -> (BN, CE) used
for max_bn in MAX_BN_LIST:
    if max_bn is None:
        BN_w, CE_w = BN_base, CE_base
        label = f"full (<{cutoff_BN:.0f})"
    else:
        w = BN_base < max_bn
        BN_w, CE_w = BN_base[w], CE_base[w]
        label = f"max BN {max_bn}"
    knee_bn, knee_ce, ipa = kneedle_elbow(BN_w, CE_w)
    windows[label] = (BN_w, CE_w, knee_bn)
    results.append({"window": label, "n_points": len(BN_w),
                    "min(CE) anchor": float(np.min(CE_w)),
                    "knee_BN": knee_bn, "CE_learned": knee_ce, "IPA": ipa})

res_df = pd.DataFrame(results)
print("\n=== Kneedle elbow vs window length ===")
print(res_df.to_string(index=False))
same = res_df["knee_BN"].nunique(dropna=True) == 1
print(f"\nSame elbow across all windows: {same}")


P%=0  BS=1024  step cutoff at BN=270  (270 points in the pre-step window)

=== Kneedle elbow vs window length ===
     window  n_points  min(CE) anchor  knee_BN  CE_learned      IPA
 max BN 150       150        0.304443     20.0    0.565573 0.086851
 max BN 250       250        0.291038     27.0    0.490360 0.067119
full (<270)       270        0.287314     27.0    0.490360 0.067119

Same elbow across all windows: False


In [2]:
# ============================================================================
# Cell 2 — Overview plot: same curve, knees found with different max BN
# ============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WIN_COLORS = ["#d62728", "#ff7f0e", "#1f77b4", "#2ca02c", "#9467bd"]

plt.rcParams.update({"font.size": 13})
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(BN_base, CE_base, s=8, color="#aaaaaa", alpha=0.6, zorder=1,
           label=r"$\overline{CE}_{test}$ (pre-step window)")

for (label, (BN_w, CE_w, knee_bn)), color in zip(windows.items(), WIN_COLORS):
    ax.axvline(BN_w[-1], color=color, linewidth=1.0, linestyle=":", alpha=0.6)
    ax.text(BN_w[-1], CE_o + 0.06, f"window end\n{BN_w[-1]:.0f}",
            ha="center", fontsize=8, color=color)
    if np.isfinite(knee_bn):
        i = int(np.argmin(np.abs(BN_base - knee_bn)))
        ax.scatter([BN_base[i]], [CE_base[i]], s=140, color=color, marker="*",
                   zorder=6, label=f"knee for {label}:  BN={knee_bn:.0f}")
        ax.axvline(knee_bn, color=color, linewidth=1.2, linestyle="--", alpha=0.7)

ax.axhline(CE_o, color="#888888", linewidth=1.0, linestyle=":")
ax.set_xlabel("Batch Number (BN)")
ax.set_ylabel("CE_TEST")
ax.set_title(f"Kneedle knee vs window length  |  P%={P_LEVEL*100:.0f}  BS={BS}  (same data)")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=10, frameon=False, loc="upper right")

out_png = os.path.join(OUT_DIR, f"kneedle_maxBN_experiment_p_{P_LEVEL}_bs_{BS}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\kneedle_maxBN_experiment_p_0.0_bs_1024.png


In [3]:
# ============================================================================
# Cell 3 — Why the knee moves: per-window Kneedle geometry
#
# One panel per window: normalized transformed curve y_t = 1 - y_hat,
# diagonal y = x, difference curve D = y_t - x_hat, knee marked.
# Truncating at a different max BN changes BOTH normalizations
# (x by the window end, y by the window min), so D and its peak shift.
# ============================================================================
fig, axes = plt.subplots(1, len(windows), figsize=(5.5 * len(windows), 4.6), sharey=True)
if len(windows) == 1:
    axes = [axes]

for ax, ((label, (BN_w, CE_w, knee_bn)), color) in zip(axes, zip(windows.items(), WIN_COLORS)):
    if len(BN_w) < 3 or np.ptp(CE_w) <= 1e-10:
        ax.set_title(f"{label} — degenerate")
        continue
    x_k = (BN_w - BN_w.min()) / np.ptp(BN_w)
    y_k = (CE_w - CE_w.min()) / np.ptp(CE_w)
    y_t  = 1.0 - y_k
    D_kn = y_t - x_k

    ax.plot(x_k, y_t, color="#1f77b4", lw=1.6, zorder=3, label=r"$\hat{y}_t = 1-\hat{y}$")
    ax.plot([0, 1], [0, 1], color="#555555", lw=1.0, ls="--", zorder=2, label=r"$y=x$")
    ax.plot(x_k, D_kn, color=color, lw=1.5, zorder=2, label=r"$D=\hat{y}_t-\hat{x}$")
    if np.isfinite(knee_bn):
        i = int(np.argmin(np.abs(BN_w - knee_bn)))
        ax.scatter([x_k[i]], [y_t[i]], s=90, color=color, marker="*", zorder=5)
        ax.scatter([x_k[i]], [D_kn[i]], s=60, color=color, marker="D",
                   facecolors="none", zorder=5)
        ax.axvline(x_k[i], color=color, lw=0.8, ls=":", alpha=0.7)
        ax.set_title(f"{label}\nknee BN={knee_bn:.0f},  peak D={D_kn[i]:.3f}", fontsize=11)
    else:
        ax.set_title(f"{label}\nno knee found", fontsize=11)
    ax.set_xlabel(r"$\hat{x}$ (BN normalized in window)")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, frameon=False, loc="center right")
axes[0].set_ylabel(r"$\hat{y}_t$,  $D$")
fig.suptitle(f"Kneedle geometry per window  |  P%={P_LEVEL*100:.0f}  BS={BS}", fontsize=13)

out_png = os.path.join(OUT_DIR, f"kneedle_maxBN_experiment_geometry_p_{P_LEVEL}_bs_{BS}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")


Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\kneedle_maxBN_experiment_geometry_p_0.0_bs_1024.png


In [4]:
# ============================================================================
# Cell 4 — NEW METHOD: elbow via minimum distance from the origin
#
# In normalized space the origin (0, 0) is the ideal corner (earliest BN,
# fully-converged CE). The elbow is the data point CLOSEST to that corner:
#     elbow = argmin_i  ( x_hat_i^2 + y_hat_i^2 )
# Unlike Kneedle, no diagonal and no far endpoint enters the selection rule —
# the window end only enters through the x normalization scale.
#
# Normalization:
#   x_hat = (BN - BN[0]) / (BN[-1] - BN[0])
#   y_hat = (CE - A) / (CE[0] - A),  A = min(CE) or mean of last ORIGIN_TAIL_N
# ============================================================================
ORIGIN_TAIL_N = None    # None = A = min(CE); int = A = mean of last N points

def origin_elbow(BN, CE, tail_n=ORIGIN_TAIL_N):
    """Min-distance-from-origin elbow. Returns (knee_BN, knee_CE, IPA) or NaNs."""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < 3 or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    if tail_n is not None:
        k = int(min(tail_n, len(CE)))
        A = float(np.mean(CE[-k:]))
    else:
        A = float(np.min(CE))
    BN_range = BN[-1] - BN[0]
    CE_range = CE[0] - A
    if BN_range <= 0 or CE_range <= 1e-10:
        return np.nan, np.nan, np.nan
    x_hat = (BN - BN[0]) / BN_range
    y_hat = (CE - A) / CE_range
    i = int(np.argmin(x_hat**2 + y_hat**2))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


results_origin = []
for label, (BN_w, CE_w, _) in windows.items():
    knee_bn, knee_ce, ipa = origin_elbow(BN_w, CE_w)
    results_origin.append({"window": label, "n_points": len(BN_w),
                           "knee_BN": knee_bn, "CE_learned": knee_ce, "IPA": ipa})

res_origin_df = pd.DataFrame(results_origin)
print("=== Min-distance-from-origin elbow vs window length ===")
print(res_origin_df.to_string(index=False))
same_o = res_origin_df["knee_BN"].nunique(dropna=True) == 1
print(f"\nSame elbow across all windows: {same_o}")


=== Min-distance-from-origin elbow vs window length ===
     window  n_points  knee_BN  CE_learned      IPA
 max BN 150       150     19.0    0.580041 0.090660
 max BN 250       250     25.0    0.507444 0.071806
full (<270)       270     27.0    0.490360 0.067119

Same elbow across all windows: False


In [5]:
# ============================================================================
# Cell 5 — COMPARISON: Kneedle vs min-distance-from-origin across windows
#
# Left: knee_BN per window for both methods (window-stability check).
# Right: normalized geometry of the FULL window — Kneedle diagonal view vs
#        origin-distance view (iso-distance circles), both elbows marked.
# ============================================================================
cmp_df = res_df[["window", "knee_BN", "IPA"]].merge(
    res_origin_df[["window", "knee_BN", "IPA"]], on="window",
    suffixes=("_kneedle", "_origin"))
print("=== Side by side ===")
print(cmp_df.to_string(index=False))
print(f"\nknee spread across windows —  Kneedle: "
      f"{res_df['knee_BN'].max() - res_df['knee_BN'].min():.0f} BN,   "
      f"origin-distance: {res_origin_df['knee_BN'].max() - res_origin_df['knee_BN'].min():.0f} BN")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: knee_BN vs window
xpos = np.arange(len(cmp_df))
ax1.plot(xpos, cmp_df["knee_BN_kneedle"], "s--", color="#ff7f0e", ms=8, lw=1.6,
         label="Kneedle")
ax1.plot(xpos, cmp_df["knee_BN_origin"], "o-", color="#2ca02c", ms=8, lw=2,
         label="min dist. from origin")
for x, r in zip(xpos, cmp_df.itertuples()):
    ax1.annotate(f"{r.knee_BN_kneedle:.0f}", (x, r.knee_BN_kneedle),
                 textcoords="offset points", xytext=(0, 8), color="#ff7f0e", fontsize=9)
    ax1.annotate(f"{r.knee_BN_origin:.0f}", (x, r.knee_BN_origin),
                 textcoords="offset points", xytext=(0, -14), color="#2ca02c", fontsize=9)
ax1.set_xticks(xpos)
ax1.set_xticklabels(cmp_df["window"])
ax1.set_xlabel("window (max BN)")
ax1.set_ylabel("knee BN")
ax1.set_title("Elbow location vs window length")
ax1.grid(True, alpha=0.3)
ax1.legend(frameon=False, fontsize=10)

# Right panel: geometry on the full window
label_full, (BN_w, CE_w, knee_kn_full) = list(windows.items())[-1]
x_hat = (BN_w - BN_w[0]) / (BN_w[-1] - BN_w[0])
A_o   = float(np.min(CE_w))
y_hat = (CE_w - A_o) / (CE_w[0] - A_o)
kn_o  = origin_elbow(BN_w, CE_w)

ax2.plot(x_hat, y_hat, color="#1f77b4", lw=1.6, zorder=3, label="normalized data")
ax2.plot([0, 1], [1, 0], color="#555555", lw=1.0, ls="--", zorder=2,
         label="Kneedle diagonal")
theta = np.linspace(0, np.pi / 2, 100)
for r in (0.1, 0.2, 0.3, 0.4):
    ax2.plot(r * np.cos(theta), r * np.sin(theta), color="#2ca02c",
             lw=0.7, ls=":", alpha=0.6, zorder=1)
if np.isfinite(kn_o[0]):
    i_o = int(np.argmin(np.abs(BN_w - kn_o[0])))
    ax2.scatter([x_hat[i_o]], [y_hat[i_o]], s=120, color="#2ca02c", marker="o",
                zorder=5, label=f"origin method  BN={kn_o[0]:.0f}")
    ax2.plot([0, x_hat[i_o]], [0, y_hat[i_o]], color="#2ca02c", lw=1.0, ls="-", alpha=0.7)
if np.isfinite(knee_kn_full):
    i_k = int(np.argmin(np.abs(BN_w - knee_kn_full)))
    ax2.scatter([x_hat[i_k]], [y_hat[i_k]], s=140, color="#ff7f0e", marker="D",
                zorder=5, label=f"Kneedle  BN={knee_kn_full:.0f}")
ax2.scatter([0], [0], s=50, color="#2ca02c", marker="+", zorder=4)
ax2.set_xlabel(r"$\hat{x}$")
ax2.set_ylabel(r"$\hat{y}$")
ax2.set_title(f"Geometry, {label_full}: circles = iso-distance from origin")
ax2.set_xlim(-0.05, 1.02); ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, alpha=0.25)
ax2.legend(frameon=False, fontsize=9)

fig.suptitle(f"Kneedle vs min-distance-from-origin  |  P%={P_LEVEL*100:.0f}  BS={BS}", fontsize=13)
out_png = os.path.join(OUT_DIR, f"kneedle_vs_origin_maxBN_p_{P_LEVEL}_bs_{BS}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved: {out_png}")


=== Side by side ===
     window  knee_BN_kneedle  IPA_kneedle  knee_BN_origin  IPA_origin
 max BN 150             20.0     0.086851            19.0    0.090660
 max BN 250             27.0     0.067119            25.0    0.071806
full (<270)             27.0     0.067119            27.0    0.067119

knee spread across windows —  Kneedle: 7 BN,   origin-distance: 8 BN

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\kneedle_vs_origin_maxBN_p_0.0_bs_1024.png


In [25]:
# ============================================================================
# Cell 6 — METHOD 3: maximum second derivative (raw data only)
#
# Classic heuristic: elbow = data point with maximum discrete second
# derivative (sharpest bend). Computed DIRECTLY on the raw data points —
# no fitted curve, and by default no smoothing (SMOOTH_W = 1):
#     d2_i = ( y_{i+1} - 2*y_i + y_{i-1} ) / dx^2      (central difference)
# Note: on a uniform BN grid the argmax is invariant to axis normalization,
# so raw CE values are used as-is.
#
# CAVEAT (visible in the results): the averaged CE decays with monotonically
# decreasing curvature, so the true maximum sits at the very start of the
# window; on raw noisy data the argmax can also be grabbed by a noise spike.
# Set SMOOTH_W to an odd number > 1 to see the effect of light smoothing.
# ============================================================================
SMOOTH_W = 30   # 1 = raw data (no smoothing); odd int > 1 = moving average

def second_derivative_elbow(BN, CE, smooth_w=SMOOTH_W):
    """Max-second-derivative elbow on the raw data.
    Returns (knee_BN, knee_CE, IPA) or NaNs."""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < max(5, smooth_w + 2) or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan
    if smooth_w > 1:
        k = np.ones(int(smooth_w)) / smooth_w
        y = np.convolve(CE, k, mode="same")
        h = int(smooth_w) // 2          # convolution edges are unreliable
    else:
        y, h = CE, 0
    d2 = np.gradient(np.gradient(y, BN), BN)   # discrete second derivative
    lo, hi = 1 + h, len(BN) - 1 - h            # exclude endpoints (+ edges)
    if hi <= lo:
        return np.nan, np.nan, np.nan
    i = lo + int(np.argmax(d2[lo:hi]))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


results_d2 = []
for label, (BN_w, CE_w, _) in windows.items():
    knee_bn, knee_ce, ipa = second_derivative_elbow(BN_w, CE_w)
    results_d2.append({"window": label, "n_points": len(BN_w),
                       "knee_BN": knee_bn, "CE_learned": knee_ce, "IPA": ipa})

res_d2_df = pd.DataFrame(results_d2)
tag = "raw data, no smoothing" if SMOOTH_W <= 1 else f"smooth_w={SMOOTH_W}"
print(f"=== Max second derivative elbow vs window length ({tag}) ===")
print(res_d2_df.to_string(index=False))
same_d2 = res_d2_df["knee_BN"].nunique(dropna=True) == 1
print(f"\nSame elbow across all windows: {same_d2}")


=== Max second derivative elbow vs window length (smooth_w=30) ===
     window  n_points  knee_BN  CE_learned      IPA
 max BN 150       150     17.0    0.614178 0.099318
 max BN 250       250     17.0    0.614178 0.099318
full (<270)       270     17.0    0.614178 0.099318

Same elbow across all windows: True


In [27]:
# ============================================================================
# Cell 7 — ALL METHODS: window-stability comparison
# Kneedle vs min-distance-from-origin vs max second derivative
# ============================================================================
all_df = res_df[["window", "knee_BN", "IPA"]].rename(
             columns={"knee_BN": "knee_kneedle", "IPA": "IPA_kneedle"}) \
         .merge(res_origin_df[["window", "knee_BN", "IPA"]].rename(
             columns={"knee_BN": "knee_origin", "IPA": "IPA_origin"}), on="window") \
         .merge(res_d2_df[["window", "knee_BN", "IPA"]].rename(
             columns={"knee_BN": "knee_2nd_deriv", "IPA": "IPA_2nd_deriv"}), on="window")
print("=== All methods ===")
print(all_df.to_string(index=False))
print("\nknee spread across windows (max - min):")
for name, col in [("Kneedle", "knee_kneedle"), ("origin", "knee_origin"),
                  ("2nd derivative", "knee_2nd_deriv")]:
    print(f"  {name:>15}: {all_df[col].max() - all_df[col].min():.0f} BN")

METHODS = [("Kneedle",               "knee_kneedle",   "#ff7f0e", "s--"),
           ("min dist. from origin", "knee_origin",    "#2ca02c", "o-"),
           ("max 2nd derivative",    "knee_2nd_deriv", "#9467bd", "^-.")]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: knee location per window per method
xpos = np.arange(len(all_df))
for name, col, color, style in METHODS:
    ax1.plot(xpos, all_df[col], style, color=color, ms=8, lw=1.8, label=name)
    for x, v in zip(xpos, all_df[col]):
        if np.isfinite(v):
            ax1.annotate(f"{v:.0f}", (x, v), textcoords="offset points",
                         xytext=(6, 4), color=color, fontsize=8)
ax1.set_xticks(xpos)
ax1.set_xticklabels(all_df["window"])
ax1.set_xlabel("window (max BN)")
ax1.set_ylabel("knee BN")
ax1.set_title("Elbow location vs window length")
ax1.grid(True, alpha=0.3)
ax1.legend(frameon=False, fontsize=9)

# Right: all elbows on the raw curve (full window)
label_full, (BN_w, CE_w, _) = list(windows.items())[-1]
ax2.scatter(BN_base, CE_base, s=8, color="#aaaaaa", alpha=0.6, zorder=1)
full_knees = [("Kneedle",               kneedle_elbow(BN_w, CE_w)[0],          "#ff7f0e", "D"),
              ("min dist. from origin", origin_elbow(BN_w, CE_w)[0],           "#2ca02c", "o"),
              ("max 2nd derivative",    second_derivative_elbow(BN_w, CE_w)[0], "#9467bd", "^")]
for name, kbn, color, marker in full_knees:
    if np.isfinite(kbn):
        i = int(np.argmin(np.abs(BN_base - kbn)))
        ax2.scatter([BN_base[i]], [CE_base[i]], s=130, color=color, marker=marker,
                    zorder=5, label=f"{name}:  BN={kbn:.0f}")
        ax2.axvline(kbn, color=color, lw=1.0, ls="--", alpha=0.6)
ax2.set_xlabel("Batch Number (BN)")
ax2.set_ylabel("CE_TEST")
ax2.set_title(f"Elbows on the full window ({label_full})")
ax2.set_xlim(0, min(150, BN_base.max()))
ax2.grid(True, alpha=0.25)
ax2.legend(frameon=False, fontsize=9)

fig.suptitle(f"Three elbow methods  |  P%={P_LEVEL*100:.0f}  BS={BS}", fontsize=13)
out_png = os.path.join(OUT_DIR, f"elbow_methods_maxBN_p_{P_LEVEL}_bs_{BS}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved: {out_png}")


=== All methods ===
     window  knee_kneedle  IPA_kneedle  knee_origin  IPA_origin  knee_2nd_deriv  IPA_2nd_deriv
 max BN 150          20.0     0.086851         19.0    0.090660            17.0       0.099318
 max BN 250          27.0     0.067119         25.0    0.071806            17.0       0.099318
full (<270)          27.0     0.067119         27.0    0.067119            17.0       0.099318

knee spread across windows (max - min):
          Kneedle: 7 BN
           origin: 8 BN
   2nd derivative: 0 BN

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\elbow_methods_maxBN_p_0.0_bs_1024.png
